In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
import os

!pip install -q -U "transformers>=4.50" "trl>=0.12" "datasets>=2.20" \
    "bitsandbytes>=0.43" accelerate sentencepiece rouge_score sacrebleu evaluate
!pip install -q "peft==0.13.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 97.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 22.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, b

In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("Please add your HF token as a Kaggle secret named HF_TOKEN")
login(HF_TOKEN)

In [7]:
import torch, gc
try: del trainer, model
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

In [ ]:

TRANSLATED_CSV   = "/kaggle/input/datasets/limphosemakale/train-data/srh_translated.csv"
EXTRA_NATIVE_CSV = "/kaggle/input/datasets/limphosemakale/native/train_df.csv"

BASE_MODEL   = "google/gemma-3-1b-it"       
OUTPUT_DIR   = "/kaggle/working/gemma3-1b-srh-lora"
MAX_SEQ_LEN  = 512
SEED         = 42

PER_LANG_CAP = {"Eng": 4000, "Kin": 8000, "Swa": 8000, "Lug": 1000}

In [10]:
import pandas as pd, re, numpy as np

df = pd.read_csv(TRANSLATED_CSV)
print("Loaded:", df.shape)

# Each row holds the same Q&A in 3 languages -> explode into one example per language.
def reshape(df):
    rows = []
    cols = {
        "Eng": ("question_en",  "answer_en"),
        "Kin": ("question_kin", "answer_kin"),
        "Swa": ("question_swa", "answer_swa"),
    }
    for _, r in df.iterrows():
        for lang, (qc, ac) in cols.items():
            q, a = r.get(qc), r.get(ac)
            if isinstance(q, str) and isinstance(a, str) and q.strip() and a.strip():
                rows.append({"lang": lang, "question": q.strip(), "answer": a.strip()})
    return pd.DataFrame(rows)

data = reshape(df)

if EXTRA_NATIVE_CSV:
    extra = pd.read_csv(EXTRA_NATIVE_CSV)
    extra = extra.rename(columns={"instruction": "question", "response": "answer"})
    extra = extra[["lang", "question", "answer"]]
    data = pd.concat([data, extra], ignore_index=True)

data = data[data["lang"].isin(["Eng", "Kin", "Swa", "Lug"])].reset_index(drop=True)
print(data["lang"].value_counts())

Loaded: (9425, 7)
lang
Eng    19129
Swa    11495
Kin    10507
Lug     3383
Name: count, dtype: int64


In [11]:
ARTIFACTS = [
    r"as of my last knowledge update",
    r"as of my last update",
    r"I('m| am) (just )?an? (AI|language model)[^.]*\.",
]
def clean(t):
    for pat in ARTIFACTS:
        t = re.sub(pat, "", t, flags=re.IGNORECASE)
    return re.sub(r"\s{2,}", " ", t).strip()

data["answer"]   = data["answer"].map(clean)
data["question"] = data["question"].map(clean)
data = data[(data["question"] != "") & (data["answer"] != "")]
data = data.drop_duplicates(subset=["lang", "question", "answer"]).reset_index(drop=True)
print("After cleaning/dedup:", data.shape)

After cleaning/dedup: (33787, 3)


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

parts = []
for lang, cap in PER_LANG_CAP.items():
    sub = data[data["lang"] == lang]
    if len(sub) > cap:
        sub = sub.sample(cap, random_state=SEED)
    parts.append(sub)
balanced = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Balanced counts:\n", balanced["lang"].value_counts())

def to_messages(row):
    return [
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["answer"]},
    ]
balanced["messages"] = balanced.apply(to_messages, axis=1)

def n_tokens(msgs):
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return len(tokenizer(text).input_ids)

lens = balanced["messages"].map(n_tokens)
print(f"\nToken lengths: median={int(lens.median())}, 95th={int(lens.quantile(.95))}, max={int(lens.max())}")
print("Rows over MAX_SEQ_LEN:", int((lens > MAX_SEQ_LEN).sum()))
print("\n--- sample formatted example ---\n",
      tokenizer.apply_chat_template(balanced["messages"].iloc[0], tokenize=False, add_generation_prompt=False)[:600])

In [ ]:
from datasets import Dataset
train_df = balanced.sample(frac=0.95, random_state=SEED)
eval_df  = balanced.drop(train_df.index)
train_ds = Dataset.from_pandas(train_df[["messages"]], preserve_index=False)
eval_ds  = Dataset.from_pandas(eval_df[["messages"]],  preserve_index=False)
print(len(train_ds), "train /", len(eval_ds), "eval")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map={"": 0},
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.enable_input_require_grads()   

from collections import Counter
print("param dtypes:", Counter(str(p.dtype) for p in model.parameters()))

lora = LoraConfig(
    r=16, lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       
    learning_rate=2e-4,                   
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="steps", eval_steps=200,
    save_strategy="steps", save_steps=200, save_total_limit=2,
    fp16=False, bf16=True,               
    max_length=MAX_SEQ_LEN,
    packing=False,
    assistant_only_loss=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none", seed=SEED,
)
trainer = SFTTrainer(
    model=model, args=cfg, peft_config=lora,
    train_dataset=train_ds, eval_dataset=eval_ds,
    processing_class=tokenizer,
)
print("grad ckpt active:", trainer.model.is_gradient_checkpointing)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter ->", OUTPUT_DIR)

In [ ]:

from transformers import pipeline
model.config.use_cache = True
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

tests = [
    "What are the symptoms of an STI?",
    "Ni ibihe bimenyetso by'indwara zandurira mu mibonano mpuzabitsina?",
    "Dalili za magonjwa ya zinaa ni zipi?",
]
for q in tests:
    prompt = tokenizer.apply_chat_template([{"role":"user","content":q}],
                                           tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, max_new_tokens=200, do_sample=False)[0]["generated_text"]
    print("Q:", q, "\nA:", out[len(prompt):].strip(), "\n", "-"*60)

In [ ]:
# Build eval set from the held-out split
eval_set = eval_df[["lang", "question", "answer"]].reset_index(drop=True)
print("Eval rows per language:")
print(eval_set["lang"].value_counts())

In [16]:
import peft
print(peft.__version__)

0.18.1


In [1]:
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 76.1 MB/s eta 0:00:00:00:01:01


In [2]:
from peft import PeftModel

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL  = "google/gemma-3-1b-it"
ADAPTER_DIR = "/kaggle/working/gemma3-1b-srh-lora"   # or a /kaggle/input/... path if reloading

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb,
    torch_dtype=torch.bfloat16, device_map={"": 0},
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)   # model object first, path second
model.eval()
model.config.use_cache = True
print("Loaded")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Loaded


In [13]:
@torch.no_grad()
def ask(question,max_new_tokens=256,temperature=0.0):
    prompt = tokenizer.apply_chat_template(
        [{"role":"user","content":question}],
        tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt,return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature>0),
        temperature= temperature if temperature > 0 else None,
        pad_token_id = tokenizer.eos_token_id
        
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:],skip_special_tokens=True)

In [14]:
print(ask("What are the symptoms of an STI?"))

The symptoms of a sexually transmitted infection (STI) can vary from person to person and can include a variety of symptoms. It is important to note that while some STIs can cause symptoms, they do not always indicate a serious infection. If you suspect you may have an STI, it's essential to seek medical attention immediately. A healthcare provider can perform a thorough examination, collect a sample of the vaginal fluid for testing, and provide an accurate diagnosis.


In [15]:
print(ask("Ni ibihe bimenyetso by'indwara zandurira mu mibonano mpuzabitsina?"))

Iyo umuntu yanduye virusi itera SIDA, umuntu ashobora kuba ufite ubwandu bwa virusi itera SIDA. Iyo umuntu ashobora kwandura virusi itera SIDA, ni iby'ingenzi cyane ko umuntu ashobora kwandura virusi itera SIDA mu gihe cy'imibonano mpuzabitsina idakingiwe. Iyo umuntu ashobora kwandura virusi itera SIDA mu gihe cy'imibonano mpuzabitsina idakingiwe, ariko hari ibyago byo kwandura virusi itera SIDA bishobora kugira ingaruka ku mibonano mpuzabitsina.
